# T-03: Speaker Identification and Voice-Activity Detection

## Narrow project question

In three continuous 300-second excerpts from the AMI Meeting Corpus, how can a simple inference-only pipeline combining pretrained Silero VAD and ECAPA-TDNN detect speech and identify the active speaker among 12 enrolled speakers?

The project uses closed-set speaker identification. Every single-speaker interval considered for speaker accuracy must belong to one of the 12 enrolled speakers.

In [38]:
%pip install -q speechbrain==1.1.0 silero-vad==6.2.1 soundfile==0.14.0

In [39]:
from pathlib import Path
import urllib.request
import math
from IPython.display import Audio, display
import numpy as np

import soundfile as sf
import torch
import torchaudio

from silero_vad import get_speech_timestamps, load_silero_vad
from speechbrain.inference.classifiers import EncoderClassifier

## 1. Download the AMI meeting data

This project uses three meetings from the AMI Meeting Corpus: `ES2002d`, `ES2008d`, and `ES2014d`. For each meeting, the mono Mix-Headset recording and its RTTM speaker annotations are downloaded.

The audio recordings contain natural meeting speech, silence, and overlapping speech. The RTTM files provide the start time, duration, and speaker identity of each annotated speech segment. Across the three selected meetings, there are 12 meeting-speaker identities.

In [40]:
MEETING_IDS = [
    "ES2002d",
    "ES2008d",
    "ES2014d"]

DATA_DIR = Path("/content/ami_subset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

meeting_data = {}

for meeting_id in MEETING_IDS:
    audio_path = (
        DATA_DIR
        / f"{meeting_id}.Mix-Headset.wav"
    )

    rttm_path = (
        DATA_DIR
        / f"{meeting_id}.rttm"
    )

    audio_url = (
        f"https://groups.inf.ed.ac.uk/ami/"
        f"AMICorpusMirror/amicorpus/"
        f"{meeting_id}/audio/"
        f"{meeting_id}.Mix-Headset.wav"
    )

    rttm_url = (
        f"https://raw.githubusercontent.com/"
        f"BUTSpeechFIT/AMI-diarization-setup/"
        f"main/only_words/rttms/train/"
        f"{meeting_id}.rttm"
    )

    if not audio_path.exists():
        print(
            f"Downloading audio for {meeting_id}..."
        )
        urllib.request.urlretrieve(
            audio_url,
            audio_path,
        )

    if not rttm_path.exists():
        print(
            f"Downloading RTTM for {meeting_id}..."
        )
        urllib.request.urlretrieve(
            rttm_url,
            rttm_path,
        )

    meeting_data[meeting_id] = {
        "audio_path": audio_path,
        "rttm_path": rttm_path,
    }

    audio_size_mb = (
        audio_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{meeting_id}: ready, "
        f"audio size {audio_size_mb:.2f} MB"
    )

print(
    f"\nPrepared {len(meeting_data)} meetings."
)

ES2002d: ready, audio size 80.08 MB
ES2008d: ready, audio size 80.13 MB
ES2014d: ready, audio size 88.85 MB

Prepared 3 meetings.


## 2. Load the RTTM annotations

Each RTTM annotation specifies the meeting, speaker identity, speech start time, and speech duration. These annotations are used to select clean enrollment segments and to create the reference timeline for evaluation.

In [41]:
def load_rttm(rttm_path, meeting_id):
    segments = []

    with open(rttm_path, "r", encoding="utf-8") as file:
        for line in file:
            fields = line.strip().split()

            if len(fields) < 8 or fields[0] != "SPEAKER":
                continue

            start = float(fields[3])
            duration = float(fields[4])
            speaker = fields[7]

            segments.append(
                {
                    "meeting": meeting_id,
                    "speaker": speaker,
                    "speaker_label": f"{meeting_id}/{speaker}",
                    "start": start,
                    "end": start + duration,
                    "duration": duration,
                }
            )

    return sorted(segments, key=lambda segment: segment["start"])


annotations_by_meeting = {}

for meeting_id in MEETING_IDS:
    rttm_path = DATA_DIR / f"{meeting_id}.rttm"

    segments = load_rttm(rttm_path, meeting_id)
    annotations_by_meeting[meeting_id] = segments

    speakers = sorted({segment["speaker"] for segment in segments})

    print(f"{meeting_id}:")
    print(f"  Annotated segments: {len(segments)}")
    print(f"  Speakers ({len(speakers)}): {speakers}")


total_segments = sum(
    len(segments)
    for segments in annotations_by_meeting.values()
)

total_speakers = sum(
    len({segment["speaker"] for segment in segments})
    for segments in annotations_by_meeting.values()
)

print()
print(f"Total annotated segments: {total_segments}")
print(f"Total meeting-speaker identities: {total_speakers}")

ES2002d:
  Annotated segments: 774
  Speakers (4): ['FEE005', 'MEE006', 'MEE007', 'MEE008']
ES2008d:
  Annotated segments: 757
  Speakers (4): ['FEE029', 'FEE030', 'FEE032', 'MEE031']
ES2014d:
  Annotated segments: 695
  Speakers (4): ['FEE055', 'MEE053', 'MEE054', 'MEE056']

Total annotated segments: 2226
Total meeting-speaker identities: 12


## 3. Select clean enrollment segments

Five clean enrollment segments are selected for each speaker. An enrollment segment must be between 0.8 and 10 seconds long, must not overlap with speech from another speaker, and must end before the continuous evaluation interval begins.

The evaluation interval starts at 2100 seconds, so enrollment and evaluation data are temporally separated. The enrollment segments are used only to create speaker centroids. Final speaker-identification evaluation will be performed on the actual speech intervals detected by Silero VAD in continuous audio.

In [42]:
MIN_ENROLLMENT_DURATION = 0.8
MAX_ENROLLMENT_DURATION = 10.0
ENROLLMENT_SEGMENTS_PER_SPEAKER = 5

EVALUATION_START = 2100.0
EVALUATION_END = 2400.0


def overlaps_another_speaker(target_segment, all_segments):
    for other_segment in all_segments:
        if other_segment["speaker"] == target_segment["speaker"]:
            continue

        overlap_start = max(
            target_segment["start"],
            other_segment["start"],
        )
        overlap_end = min(
            target_segment["end"],
            other_segment["end"],
        )

        if overlap_end > overlap_start:
            return True

    return False


clean_candidates_by_speaker = {}
enrollment_segments = []

for meeting_id in MEETING_IDS:
    meeting_segments = annotations_by_meeting[meeting_id]
    meeting_speakers = sorted(
        {segment["speaker"] for segment in meeting_segments}
    )

    print(f"{meeting_id}:")

    for speaker in meeting_speakers:
        speaker_label = f"{meeting_id}/{speaker}"

        candidates = [
            segment
            for segment in meeting_segments
            if segment["speaker"] == speaker
            and MIN_ENROLLMENT_DURATION
            <= segment["duration"]
            <= MAX_ENROLLMENT_DURATION
            and segment["end"] <= EVALUATION_START
            and not overlaps_another_speaker(
                segment,
                meeting_segments,
            )
        ]

        candidates = sorted(
            candidates,
            key=lambda segment: segment["start"],
        )

        clean_candidates_by_speaker[speaker_label] = candidates

        if len(candidates) < ENROLLMENT_SEGMENTS_PER_SPEAKER:
            raise ValueError(
                f"Not enough clean enrollment segments for "
                f"{speaker_label}: {len(candidates)} found."
            )

        selected_segments = candidates[
            :ENROLLMENT_SEGMENTS_PER_SPEAKER
        ]
        enrollment_segments.extend(selected_segments)

        print(
            f"  {speaker}: "
            f"{len(candidates)} clean candidates, "
            f"{len(selected_segments)} selected"
        )


enrollment_speakers = sorted(
    {segment["speaker_label"] for segment in enrollment_segments}
)

enrollment_durations = [
    segment["duration"] for segment in enrollment_segments
]

latest_enrollment_end = max(
    segment["end"] for segment in enrollment_segments
)

print()
print(f"Total enrollment segments: {len(enrollment_segments)}")
print(f"Total enrolled speakers: {len(enrollment_speakers)}")
print(
    "Enrollment duration range: "
    f"{min(enrollment_durations):.2f} to "
    f"{max(enrollment_durations):.2f} seconds"
)
print(
    f"Latest enrollment end time: "
    f"{latest_enrollment_end:.2f} seconds"
)
print(
    f"Evaluation interval: "
    f"{EVALUATION_START:.0f} to "
    f"{EVALUATION_END:.0f} seconds"
)

ES2002d:
  FEE005: 24 clean candidates, 5 selected
  MEE006: 10 clean candidates, 5 selected
  MEE007: 14 clean candidates, 5 selected
  MEE008: 20 clean candidates, 5 selected
ES2008d:
  FEE029: 42 clean candidates, 5 selected
  FEE030: 38 clean candidates, 5 selected
  FEE032: 21 clean candidates, 5 selected
  MEE031: 12 clean candidates, 5 selected
ES2014d:
  FEE055: 16 clean candidates, 5 selected
  MEE053: 27 clean candidates, 5 selected
  MEE054: 8 clean candidates, 5 selected
  MEE056: 33 clean candidates, 5 selected

Total enrollment segments: 60
Total enrolled speakers: 12
Enrollment duration range: 0.82 to 9.92 seconds
Latest enrollment end time: 1826.23 seconds
Evaluation interval: 2100 to 2400 seconds


## 4. Inspect recordings and define audio extraction

The sample rate, channel count, and duration of each recording are checked before inference. To reduce memory usage, only the required enrollment or evaluation interval is read from each recording instead of loading the complete meeting into memory.

In [43]:
TARGET_SAMPLE_RATE = 16000
audio_paths = {}

for meeting_id in MEETING_IDS:
    audio_path = DATA_DIR / f"{meeting_id}.Mix-Headset.wav"
    audio_info = sf.info(str(audio_path))

    if audio_info.duration < EVALUATION_END:
        raise ValueError(
            f"{meeting_id} is shorter than the selected "
            f"evaluation interval."
        )

    audio_paths[meeting_id] = audio_path

    print(f"{meeting_id}:")
    print(f"  Sample rate: {audio_info.samplerate} Hz")
    print(f"  Channels: {audio_info.channels}")
    print(f"  Duration: {audio_info.duration:.2f} seconds")
    print(f"  Duration: {audio_info.duration / 60:.2f} minutes")


def read_audio_segment(meeting_id, start, end):
    audio_path = audio_paths[meeting_id]

    with sf.SoundFile(str(audio_path), mode="r") as audio_file:
        source_sample_rate = audio_file.samplerate

        start_frame = round(start * source_sample_rate)
        number_of_frames = round(
            (end - start) * source_sample_rate
        )

        audio_file.seek(start_frame)

        audio_array = audio_file.read(
            frames=number_of_frames,
            dtype="float32",
            always_2d=True,
        )

    waveform = torch.from_numpy(audio_array).mean(dim=1)

    if source_sample_rate != TARGET_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(
            waveform,
            source_sample_rate,
            TARGET_SAMPLE_RATE,
        )

    return waveform.contiguous()

ES2002d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2624.17 seconds
  Duration: 43.74 minutes
ES2008d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2625.82 seconds
  Duration: 43.76 minutes
ES2014d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2911.36 seconds
  Duration: 48.52 minutes


## 5. Load the pretrained baseline models

The pretrained Silero VAD model is used to detect speech intervals in continuous meeting audio. SpeechBrain's pretrained ECAPA-TDNN model is used to convert each speech interval into a speaker embedding.

Both models are used only for inference on CPU. No model training or fine-tuning is performed.

In [44]:
DEVICE = "cpu"

vad_model = load_silero_vad()

speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained_models/ecapa_voxceleb",
    run_opts={"device": DEVICE},
)

print("Silero VAD loaded successfully.")
print("SpeechBrain ECAPA-TDNN loaded successfully.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Silero VAD loaded successfully.
SpeechBrain ECAPA-TDNN loaded successfully.


## 6. Create enrolled speaker centroids

ECAPA-TDNN converts each enrollment segment into a 192-dimensional speaker embedding. The five normalized embeddings belonging to each enrolled speaker are averaged and normalized again to create one speaker centroid.

Speaker identification will later compare each VAD-detected speech interval with these 12 centroids using cosine similarity.

In [45]:
@torch.inference_mode()
def extract_speaker_embedding(waveform):
    if waveform.numel() == 0:
        raise ValueError("Cannot extract an embedding from empty audio.")

    waveform = waveform.to(DEVICE).unsqueeze(0)

    embedding = speaker_model.encode_batch(
        waveform
    ).squeeze()

    embedding = embedding.cpu()
    embedding = embedding / embedding.norm(p=2)

    return embedding


enrollment_embeddings = {}

for speaker_label in enrollment_speakers:
    speaker_segments = [
        segment
        for segment in enrollment_segments
        if segment["speaker_label"] == speaker_label
    ]

    speaker_embedding_list = []

    print(f"Processing {speaker_label}...")

    for segment in speaker_segments:
        waveform = read_audio_segment(
            meeting_id=segment["meeting"],
            start=segment["start"],
            end=segment["end"],
        )

        embedding = extract_speaker_embedding(waveform)
        speaker_embedding_list.append(embedding)

    enrollment_embeddings[speaker_label] = speaker_embedding_list


speaker_centroids = {}

for speaker_label in enrollment_speakers:
    centroid = torch.stack(
        enrollment_embeddings[speaker_label]
    ).mean(dim=0)

    centroid = centroid / centroid.norm(p=2)
    speaker_centroids[speaker_label] = centroid

    print(
        f"{speaker_label}: "
        f"{len(enrollment_embeddings[speaker_label])} segments, "
        f"centroid shape {tuple(centroid.shape)}, "
        f"norm {centroid.norm(p=2).item():.4f}"
    )


print()
print(f"Created centroids for {len(speaker_centroids)} speakers.")

Processing ES2002d/FEE005...
Processing ES2002d/MEE006...
Processing ES2002d/MEE007...
Processing ES2002d/MEE008...
Processing ES2008d/FEE029...
Processing ES2008d/FEE030...
Processing ES2008d/FEE032...
Processing ES2008d/MEE031...
Processing ES2014d/FEE055...
Processing ES2014d/MEE053...
Processing ES2014d/MEE054...
Processing ES2014d/MEE056...
ES2002d/FEE005: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE006: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE007: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE008: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE029: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE030: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE032: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/MEE031: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/FEE055: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/MEE053: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/MEE054: 5 segm

## 7. Verify the speaker-identification component

As a Phase 1 test, the speaker-identification component is applied to one clean segment that was not used for enrollment. Its ECAPA-TDNN embedding is compared with all 12 enrolled centroids using cosine similarity.

This isolated check only verifies that the component runs correctly. In Phase 2, speaker identification will instead be applied to the intervals produced by Silero VAD from continuous audio.

In [46]:
def identify_speaker(waveform):
    embedding = extract_speaker_embedding(waveform)

    similarity_scores = {
        speaker_label: torch.dot(
            embedding,
            centroid,
        ).item()
        for speaker_label, centroid
        in speaker_centroids.items()
    }

    predicted_speaker = max(
        similarity_scores,
        key=similarity_scores.get,
    )

    return predicted_speaker, similarity_scores


example_speaker = enrollment_speakers[0]

example_segment = clean_candidates_by_speaker[
    example_speaker
][ENROLLMENT_SEGMENTS_PER_SPEAKER]

example_waveform = read_audio_segment(
    meeting_id=example_segment["meeting"],
    start=example_segment["start"],
    end=example_segment["end"],
)

predicted_speaker, similarity_scores = identify_speaker(
    example_waveform
)

sorted_scores = sorted(
    similarity_scores.items(),
    key=lambda item: item[1],
    reverse=True,
)

print("Speaker-identification test:")
print(f"Meeting: {example_segment['meeting']}")
print(f"True speaker: {example_segment['speaker_label']}")
print(f"Predicted speaker: {predicted_speaker}")
print(f"Duration: {example_segment['duration']:.2f} seconds")

print()
print("Cosine similarity scores:")

for speaker_label, score in sorted_scores:
    print(f"  {speaker_label}: {score:.4f}")

Speaker-identification test:
Meeting: ES2002d
True speaker: ES2002d/FEE005
Predicted speaker: ES2002d/FEE005
Duration: 3.11 seconds

Cosine similarity scores:
  ES2002d/FEE005: 0.7257
  ES2014d/FEE055: 0.2303
  ES2014d/MEE056: 0.1795
  ES2008d/FEE030: 0.1695
  ES2002d/MEE006: 0.1236
  ES2008d/FEE029: 0.1146
  ES2014d/MEE054: 0.1008
  ES2002d/MEE007: 0.0701
  ES2008d/MEE031: 0.0548
  ES2014d/MEE053: 0.0413
  ES2002d/MEE008: 0.0381
  ES2008d/FEE032: 0.0053


## 8. Prepare the continuous evaluation excerpts

A continuous 300-second interval from 2100 to 2400 seconds is selected from each meeting. These intervals occur after all selected enrollment segments and contain speech, silence, and overlapping speech.

RTTM annotations are clipped to each evaluation interval and converted to local times between 0 and 300 seconds. They will later provide the reference for VAD evaluation and for determining which speaker or speakers are active during each detected interval.

In [47]:
EVALUATION_DURATION = EVALUATION_END - EVALUATION_START


def merge_intervals(intervals):
    if not intervals:
        return []

    sorted_intervals = sorted(
        intervals,
        key=lambda interval: interval["start"],
    )

    merged = [sorted_intervals[0].copy()]

    for interval in sorted_intervals[1:]:
        previous = merged[-1]

        if interval["start"] <= previous["end"]:
            previous["end"] = max(
                previous["end"],
                interval["end"],
            )
        else:
            merged.append(interval.copy())

    for interval in merged:
        interval["duration"] = (
            interval["end"] - interval["start"]
        )

    return merged


def calculate_timeline_durations(
    merged_intervals_by_speaker,
):
    events = []

    for speaker_intervals in (
        merged_intervals_by_speaker.values()
    ):
        for interval in speaker_intervals:
            events.append((interval["start"], 1))
            events.append((interval["end"], -1))

    events.sort()

    active_speakers = 0
    previous_time = 0.0
    speech_duration = 0.0
    overlap_duration = 0.0

    for event_time, change in events:
        interval_duration = event_time - previous_time

        if active_speakers >= 1:
            speech_duration += interval_duration

        if active_speakers >= 2:
            overlap_duration += interval_duration

        active_speakers += change
        previous_time = event_time

    return speech_duration, overlap_duration


evaluation_waveforms = {}
reference_segments_by_meeting = {}
reference_intervals_by_speaker = {}
reference_speech_intervals = {}

total_reference_speech = 0.0
total_reference_overlap = 0.0

for meeting_id in MEETING_IDS:
    evaluation_waveform = read_audio_segment(
        meeting_id=meeting_id,
        start=EVALUATION_START,
        end=EVALUATION_END,
    )

    evaluation_waveforms[meeting_id] = evaluation_waveform

    local_segments = []

    for segment in annotations_by_meeting[meeting_id]:
        clipped_start = max(
            segment["start"],
            EVALUATION_START,
        )
        clipped_end = min(
            segment["end"],
            EVALUATION_END,
        )

        if clipped_end <= clipped_start:
            continue

        local_segments.append(
            {
                "meeting": meeting_id,
                "speaker": segment["speaker"],
                "speaker_label": segment["speaker_label"],
                "start": clipped_start - EVALUATION_START,
                "end": clipped_end - EVALUATION_START,
                "duration": clipped_end - clipped_start,
            }
        )

    reference_segments_by_meeting[meeting_id] = (
        local_segments
    )

    meeting_speakers = sorted(
        {
            segment["speaker_label"]
            for segment in local_segments
        }
    )

    merged_by_speaker = {}

    for speaker_label in meeting_speakers:
        speaker_intervals = [
            {
                "start": segment["start"],
                "end": segment["end"],
            }
            for segment in local_segments
            if segment["speaker_label"] == speaker_label
        ]

        merged_by_speaker[speaker_label] = (
            merge_intervals(speaker_intervals)
        )

    reference_intervals_by_speaker[meeting_id] = (
        merged_by_speaker
    )

    all_speaker_intervals = [
        interval
        for speaker_intervals in merged_by_speaker.values()
        for interval in speaker_intervals
    ]

    meeting_speech_intervals = merge_intervals(
        all_speaker_intervals
    )

    reference_speech_intervals[meeting_id] = (
        meeting_speech_intervals
    )

    speech_duration, overlap_duration = (
        calculate_timeline_durations(merged_by_speaker)
    )

    silence_duration = (
        EVALUATION_DURATION - speech_duration
    )

    total_reference_speech += speech_duration
    total_reference_overlap += overlap_duration

    print(f"{meeting_id}:")
    print(
        f"  Waveform shape: "
        f"{tuple(evaluation_waveform.shape)}"
    )
    print(
        f"  Reference speech: "
        f"{speech_duration:.2f} seconds"
    )
    print(
        f"  Reference silence: "
        f"{silence_duration:.2f} seconds"
    )
    print(
        f"  Reference overlap: "
        f"{overlap_duration:.2f} seconds"
    )
    print(
        f"  Active speakers: {len(meeting_speakers)}"
    )


total_evaluation_duration = (
    len(MEETING_IDS) * EVALUATION_DURATION
)

print()
print(
    f"Total evaluation audio: "
    f"{total_evaluation_duration / 60:.2f} minutes"
)
print(
    f"Total reference speech: "
    f"{total_reference_speech:.2f} seconds"
)
print(
    f"Total reference silence: "
    f"{total_evaluation_duration - total_reference_speech:.2f} "
    f"seconds"
)
print(
    f"Total reference overlap: "
    f"{total_reference_overlap:.2f} seconds"
)

ES2002d:
  Waveform shape: (4800000,)
  Reference speech: 264.79 seconds
  Reference silence: 35.21 seconds
  Reference overlap: 50.79 seconds
  Active speakers: 4
ES2008d:
  Waveform shape: (4800000,)
  Reference speech: 238.45 seconds
  Reference silence: 61.55 seconds
  Reference overlap: 21.86 seconds
  Active speakers: 4
ES2014d:
  Waveform shape: (4800000,)
  Reference speech: 139.14 seconds
  Reference silence: 160.86 seconds
  Reference overlap: 10.24 seconds
  Active speakers: 4

Total evaluation audio: 15.00 minutes
Total reference speech: 642.38 seconds
Total reference silence: 257.62 seconds
Total reference overlap: 82.89 seconds


## 9. Verify the voice-activity-detection component

As a Phase 1 test, Silero VAD is applied to a separate 30-second excerpt from `ES2002d`, covering 2000 to 2030 seconds. This excerpt is outside the future evaluation interval.

The test only verifies that Silero VAD produces usable speech boundaries. No VAD metric is calculated in Phase 1, and the model's default inference settings are not tuned using the evaluation data.

In [48]:
def detect_speech_intervals(waveform):
    timestamps = get_speech_timestamps(
        waveform,
        vad_model,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_seconds=False,
    )

    detected_intervals = []

    for timestamp in timestamps:
        start_sample = int(timestamp["start"])
        end_sample = int(timestamp["end"])

        detected_intervals.append(
            {
                "start_sample": start_sample,
                "end_sample": end_sample,
                "start": (
                    start_sample / TARGET_SAMPLE_RATE
                ),
                "end": (
                    end_sample / TARGET_SAMPLE_RATE
                ),
                "duration": (
                    (end_sample - start_sample)
                    / TARGET_SAMPLE_RATE
                ),
            }
        )

    return detected_intervals


SMOKE_TEST_MEETING = "ES2002d"
SMOKE_TEST_START = 2000.0
SMOKE_TEST_END = 2030.0

vad_smoke_waveform = read_audio_segment(
    meeting_id=SMOKE_TEST_MEETING,
    start=SMOKE_TEST_START,
    end=SMOKE_TEST_END,
)

vad_smoke_intervals = detect_speech_intervals(
    vad_smoke_waveform
)

detected_speech_duration = sum(
    interval["duration"]
    for interval in vad_smoke_intervals
)

print("VAD smoke test:")
print(f"Meeting: {SMOKE_TEST_MEETING}")
print(
    f"Meeting-time interval: "
    f"{SMOKE_TEST_START:.0f} to "
    f"{SMOKE_TEST_END:.0f} seconds"
)
print(
    f"Detected speech intervals: "
    f"{len(vad_smoke_intervals)}"
)
print(
    f"Detected speech duration: "
    f"{detected_speech_duration:.2f} seconds"
)

print()
print("First five detected intervals in local excerpt time:")

for interval in vad_smoke_intervals[:5]:
    print(
        f"  {interval['start']:.2f} to "
        f"{interval['end']:.2f} seconds"
    )

VAD smoke test:
Meeting: ES2002d
Meeting-time interval: 2000 to 2030 seconds
Detected speech intervals: 9
Detected speech duration: 26.35 seconds

First five detected intervals in local excerpt time:
  0.77 to 3.58 seconds
  3.87 to 4.57 seconds
  4.83 to 5.76 seconds
  5.86 to 10.46 seconds
  11.36 to 13.92 seconds


## 10. Phase 2 evaluation plan

The two components will be connected serially in Phase 2:

1. Silero VAD will process each continuous 300-second excerpt.
2. Each detected speech interval will be extracted from the waveform.
3. ECAPA-TDNN will create an embedding for that detected interval.
4. Cosine similarity with the 12 enrolled centroids will determine the predicted speaker.

Two metrics will be reported:

- **VAD F1 score:** Reference and predicted speech activity will be compared at 10-millisecond frame resolution. Overlapping speech is still considered speech for this metric. F1 combines the ability to detect reference speech with the ability to avoid false speech detections.
- **Top-1 speaker-identification accuracy:** Accuracy will be calculated only for VAD-detected intervals that overlap speech from exactly one RTTM speaker. A prediction is correct when the highest-scoring centroid belongs to that reference speaker. Intervals containing no reference speech or more than one reference speaker will be analysed separately.

As a simple exploratory overlap heuristic, the difference between the two highest cosine-similarity scores will also be inspected. If this similarity margin is smaller than a threshold that will be selected and frozen before final evaluation, the two highest-scoring speakers will be returned as possible speakers. Otherwise, only the highest-scoring speaker will be returned.

This heuristic is not treated as a reliable overlap detector because ECAPA-TDNN produces a single embedding and was not designed as a multi-speaker classifier. Its behaviour on RTTM-annotated overlap intervals will therefore be examined only as part of the manual error analysis.

No evaluation metrics are calculated in Phase 1.

## 11. Phase 2 analysis plan

The following error categories and data slices will be inspected in Phase 2:

1. **VAD false alarms and missed speech:** Representative false-positive and false-negative regions will be inspected, especially around silence, breaths, hesitations, and low-energy speech.
2. **Short versus long detected intervals:** Eligible single-speaker intervals will be divided using their median duration, and top-1 speaker-identification accuracy will be compared between the short and long groups.
3. **Overlapping or mixed-speaker intervals:** Detected intervals associated with two or more RTTM speakers will be inspected separately. Simultaneous overlap will be distinguished from cases in which one VAD interval merges consecutive speaker turns. For intervals containing exactly two simultaneously active speakers, the analysis will examine whether the similarity-margin heuristic includes both reference speakers among its two candidates and will describe representative successes and failures.

These analyses are planned for Phase 2 and are not performed in Phase 1.

## 12. Run the serial VAD and speaker-identification pipeline

Each continuous evaluation excerpt is first processed by Silero VAD. Every detected interval is then extracted using the exact VAD sample boundaries and passed to ECAPA-TDNN. Its embedding is compared with all 12 enrolled centroids.

The highest-scoring centroid is always stored as the top-1 speaker prediction. For the exploratory overlap heuristic, the two highest-scoring speakers are returned when their cosine-similarity margin is smaller than the fixed Phase 2 threshold of 0.05. RTTM annotations are not used during prediction.

In [49]:
OVERLAP_MARGIN = 0.05


def run_serial_pipeline(meeting_id, waveform):
    detected_intervals = detect_speech_intervals(waveform)
    meeting_results = []

    print(
        f"{meeting_id}: "
        f"{len(detected_intervals)} VAD intervals detected"
    )

    for interval_index, interval in enumerate(
        detected_intervals,
        start=1,
    ):
        detected_waveform = waveform[
            interval["start_sample"]:
            interval["end_sample"]
        ]

        top1_speaker, similarity_scores = identify_speaker(
            detected_waveform
        )

        ordered_scores = sorted(
            similarity_scores.items(),
            key=lambda item: item[1],
            reverse=True,
        )

        top1_speaker, top1_score = ordered_scores[0]
        top2_speaker, top2_score = ordered_scores[1]

        similarity_margin = top1_score - top2_score

        if similarity_margin < OVERLAP_MARGIN:
            candidate_speakers = [
                top1_speaker,
                top2_speaker,
            ]
        else:
            candidate_speakers = [top1_speaker]

        meeting_results.append(
            {
                "meeting": meeting_id,
                "interval_index": interval_index,
                "start_sample": interval["start_sample"],
                "end_sample": interval["end_sample"],
                "start": interval["start"],
                "end": interval["end"],
                "duration": interval["duration"],
                "top1_speaker": top1_speaker,
                "top1_score": top1_score,
                "top2_speaker": top2_speaker,
                "top2_score": top2_score,
                "similarity_margin": similarity_margin,
                "candidate_speakers": candidate_speakers,
                "similarity_scores": similarity_scores,
            }
        )

        if (
            interval_index % 25 == 0
            or interval_index == len(detected_intervals)
        ):
            print(
                f"  Processed "
                f"{interval_index}/"
                f"{len(detected_intervals)} intervals"
            )

    return meeting_results


serial_results_by_meeting = {}

for meeting_id in MEETING_IDS:
    serial_results_by_meeting[meeting_id] = (
        run_serial_pipeline(
            meeting_id=meeting_id,
            waveform=evaluation_waveforms[meeting_id],
        )
    )

    meeting_results = serial_results_by_meeting[
        meeting_id
    ]

    single_candidate_count = sum(
        len(result["candidate_speakers"]) == 1
        for result in meeting_results
    )

    two_candidate_count = sum(
        len(result["candidate_speakers"]) == 2
        for result in meeting_results
    )

    print(
        f"  One-speaker outputs: "
        f"{single_candidate_count}"
    )
    print(
        f"  Two-speaker heuristic outputs: "
        f"{two_candidate_count}"
    )
    print()


all_serial_results = [
    result
    for meeting_results
    in serial_results_by_meeting.values()
    for result in meeting_results
]

print(
    f"Total processed VAD intervals: "
    f"{len(all_serial_results)}"
)

ES2002d: 100 VAD intervals detected
  Processed 25/100 intervals
  Processed 50/100 intervals
  Processed 75/100 intervals
  Processed 100/100 intervals
  One-speaker outputs: 83
  Two-speaker heuristic outputs: 17

ES2008d: 91 VAD intervals detected
  Processed 25/91 intervals
  Processed 50/91 intervals
  Processed 75/91 intervals
  Processed 91/91 intervals
  One-speaker outputs: 77
  Two-speaker heuristic outputs: 14

ES2014d: 65 VAD intervals detected
  Processed 25/65 intervals
  Processed 50/65 intervals
  Processed 65/65 intervals
  One-speaker outputs: 55
  Two-speaker heuristic outputs: 10

Total processed VAD intervals: 256


## 13. Match VAD intervals with RTTM reference speakers

After prediction, each VAD-detected interval is compared with the RTTM annotations for evaluation. RTTM information is not used by Silero VAD or ECAPA-TDNN during inference.

The detected intervals are divided into four reference groups:

1. `false_alarm`: no RTTM speaker overlaps the detected interval.
2. `single_speaker`: exactly one unique RTTM speaker overlaps the interval.
3. `simultaneous_overlap`: at least two RTTM speakers are active simultaneously during some part of the interval.
4. `consecutive_mixed_speakers`: the interval contains turns from multiple speakers, but they are not active simultaneously.

Only `single_speaker` intervals will be used for top-1 speaker-identification accuracy. Only RTTM-confirmed `simultaneous_overlap` intervals with exactly two reference speakers will be used to inspect the overlap heuristic.

In [50]:
def temporal_overlap_duration(
    start_a,
    end_a,
    start_b,
    end_b,
):
    return max(
        0.0,
        min(end_a, end_b) - max(start_a, start_b),
    )


def get_reference_information(
    meeting_id,
    detected_interval,
):
    detected_start = detected_interval["start"]
    detected_end = detected_interval["end"]

    meeting_reference = (
        reference_intervals_by_speaker[meeting_id]
    )

    overlap_duration_by_speaker = {}

    for speaker_label, speaker_intervals in (
        meeting_reference.items()
    ):
        total_overlap = sum(
            temporal_overlap_duration(
                detected_start,
                detected_end,
                reference_interval["start"],
                reference_interval["end"],
            )
            for reference_interval in speaker_intervals
        )

        if total_overlap > 0:
            overlap_duration_by_speaker[
                speaker_label
            ] = total_overlap

    reference_speakers = sorted(
        overlap_duration_by_speaker
    )

    simultaneous_pairs = []

    for first_index in range(
        len(reference_speakers)
    ):
        for second_index in range(
            first_index + 1,
            len(reference_speakers),
        ):
            first_speaker = reference_speakers[
                first_index
            ]
            second_speaker = reference_speakers[
                second_index
            ]

            pair_has_simultaneous_overlap = False

            for first_interval in meeting_reference[
                first_speaker
            ]:
                for second_interval in meeting_reference[
                    second_speaker
                ]:
                    simultaneous_start = max(
                        detected_start,
                        first_interval["start"],
                        second_interval["start"],
                    )

                    simultaneous_end = min(
                        detected_end,
                        first_interval["end"],
                        second_interval["end"],
                    )

                    if simultaneous_end > simultaneous_start:
                        pair_has_simultaneous_overlap = True
                        break

                if pair_has_simultaneous_overlap:
                    break

            if pair_has_simultaneous_overlap:
                simultaneous_pairs.append(
                    (
                        first_speaker,
                        second_speaker,
                    )
                )

    if len(reference_speakers) == 0:
        reference_group = "false_alarm"

    elif len(reference_speakers) == 1:
        reference_group = "single_speaker"

    elif simultaneous_pairs:
        reference_group = "simultaneous_overlap"

    else:
        reference_group = (
            "consecutive_mixed_speakers"
        )

    return {
        "reference_group": reference_group,
        "reference_speakers": reference_speakers,
        "reference_overlap_duration_by_speaker": (
            overlap_duration_by_speaker
        ),
        "simultaneous_pairs": simultaneous_pairs,
    }


REFERENCE_GROUPS = [
    "false_alarm",
    "single_speaker",
    "simultaneous_overlap",
    "consecutive_mixed_speakers",
]

for meeting_id in MEETING_IDS:
    meeting_results = serial_results_by_meeting[
        meeting_id
    ]

    for result in meeting_results:
        reference_information = (
            get_reference_information(
                meeting_id=meeting_id,
                detected_interval=result,
            )
        )

        result.update(reference_information)

    print(f"{meeting_id}:")

    for reference_group in REFERENCE_GROUPS:
        group_count = sum(
            result["reference_group"]
            == reference_group
            for result in meeting_results
        )

        print(
            f"  {reference_group}: "
            f"{group_count}"
        )

    exactly_two_overlap_count = sum(
        result["reference_group"]
        == "simultaneous_overlap"
        and len(result["reference_speakers"]) == 2
        for result in meeting_results
    )

    print(
        "  exactly_two_speaker_overlap: "
        f"{exactly_two_overlap_count}"
    )
    print()


print("Overall reference groups:")

for reference_group in REFERENCE_GROUPS:
    group_count = sum(
        result["reference_group"]
        == reference_group
        for result in all_serial_results
    )

    print(
        f"  {reference_group}: "
        f"{group_count}"
    )

overall_two_speaker_overlap = sum(
    result["reference_group"]
    == "simultaneous_overlap"
    and len(result["reference_speakers"]) == 2
    for result in all_serial_results
)

print(
    "  exactly_two_speaker_overlap: "
    f"{overall_two_speaker_overlap}"
)

ES2002d:
  false_alarm: 1
  single_speaker: 56
  simultaneous_overlap: 41
  consecutive_mixed_speakers: 2
  exactly_two_speaker_overlap: 29

ES2008d:
  false_alarm: 1
  single_speaker: 56
  simultaneous_overlap: 30
  consecutive_mixed_speakers: 4
  exactly_two_speaker_overlap: 21

ES2014d:
  false_alarm: 2
  single_speaker: 48
  simultaneous_overlap: 14
  consecutive_mixed_speakers: 1
  exactly_two_speaker_overlap: 9

Overall reference groups:
  false_alarm: 4
  single_speaker: 160
  simultaneous_overlap: 85
  consecutive_mixed_speakers: 7
  exactly_two_speaker_overlap: 59


## 14. Compute frame-level VAD F1

Each 300-second excerpt is divided into 30,000 non-overlapping 10-ms frames. A reference frame is labelled as speech if it overlaps any RTTM speech interval. A predicted frame is labelled as speech if it overlaps any interval detected by Silero VAD.

Speaker identities are ignored for this metric. Overlapping speech from multiple speakers still produces one binary speech label. Precision, recall, and F1 are calculated for each meeting and over all 90,000 evaluation frames.

In [51]:
FRAME_DURATION = 0.010

NUMBER_OF_EVALUATION_FRAMES = round(
    EVALUATION_DURATION / FRAME_DURATION
)


def intervals_to_frame_mask(intervals):
    frame_mask = torch.zeros(
        NUMBER_OF_EVALUATION_FRAMES,
        dtype=torch.bool,
    )

    for interval in intervals:
        interval_start = max(
            0.0,
            min(
                EVALUATION_DURATION,
                interval["start"],
            ),
        )

        interval_end = max(
            0.0,
            min(
                EVALUATION_DURATION,
                interval["end"],
            ),
        )

        if interval_end <= interval_start:
            continue

        first_frame = math.floor(
            interval_start / FRAME_DURATION
        )

        last_frame_exclusive = math.ceil(
            interval_end / FRAME_DURATION
        )

        first_frame = max(
            0,
            first_frame,
        )

        last_frame_exclusive = min(
            NUMBER_OF_EVALUATION_FRAMES,
            last_frame_exclusive,
        )

        frame_mask[
            first_frame:last_frame_exclusive
        ] = True

    return frame_mask


def calculate_vad_metrics(
    reference_mask,
    predicted_mask,
):
    true_positive = int(
        torch.logical_and(
            reference_mask,
            predicted_mask,
        ).sum().item()
    )

    false_positive = int(
        torch.logical_and(
            torch.logical_not(reference_mask),
            predicted_mask,
        ).sum().item()
    )

    false_negative = int(
        torch.logical_and(
            reference_mask,
            torch.logical_not(predicted_mask),
        ).sum().item()
    )

    true_negative = int(
        torch.logical_and(
            torch.logical_not(reference_mask),
            torch.logical_not(predicted_mask),
        ).sum().item()
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0.0
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


reference_vad_masks = {}
predicted_vad_masks = {}
vad_metrics_by_meeting = {}

for meeting_id in MEETING_IDS:
    reference_mask = intervals_to_frame_mask(
        reference_speech_intervals[meeting_id]
    )

    predicted_mask = intervals_to_frame_mask(
        serial_results_by_meeting[meeting_id]
    )

    reference_vad_masks[meeting_id] = (
        reference_mask
    )

    predicted_vad_masks[meeting_id] = (
        predicted_mask
    )

    meeting_metrics = calculate_vad_metrics(
        reference_mask=reference_mask,
        predicted_mask=predicted_mask,
    )

    vad_metrics_by_meeting[meeting_id] = (
        meeting_metrics
    )

    print(f"{meeting_id}:")
    print(
        f"  TP frames: "
        f"{meeting_metrics['true_positive']}"
    )
    print(
        f"  FP frames: "
        f"{meeting_metrics['false_positive']}"
    )
    print(
        f"  FN frames: "
        f"{meeting_metrics['false_negative']}"
    )
    print(
        f"  TN frames: "
        f"{meeting_metrics['true_negative']}"
    )
    print(
        f"  Precision: "
        f"{meeting_metrics['precision']:.4f}"
    )
    print(
        f"  Recall: "
        f"{meeting_metrics['recall']:.4f}"
    )
    print(
        f"  F1: "
        f"{meeting_metrics['f1']:.4f}"
    )
    print()


overall_true_positive = sum(
    metrics["true_positive"]
    for metrics in vad_metrics_by_meeting.values()
)

overall_false_positive = sum(
    metrics["false_positive"]
    for metrics in vad_metrics_by_meeting.values()
)

overall_false_negative = sum(
    metrics["false_negative"]
    for metrics in vad_metrics_by_meeting.values()
)

overall_true_negative = sum(
    metrics["true_negative"]
    for metrics in vad_metrics_by_meeting.values()
)

overall_reference_mask = torch.cat(
    [
        reference_vad_masks[meeting_id]
        for meeting_id in MEETING_IDS
    ]
)

overall_predicted_mask = torch.cat(
    [
        predicted_vad_masks[meeting_id]
        for meeting_id in MEETING_IDS
    ]
)

overall_vad_metrics = calculate_vad_metrics(
    reference_mask=overall_reference_mask,
    predicted_mask=overall_predicted_mask,
)

print("Overall VAD metrics:")
print(
    f"  Total frames: "
    f"{len(overall_reference_mask)}"
)
print(
    f"  TP frames: "
    f"{overall_true_positive}"
)
print(
    f"  FP frames: "
    f"{overall_false_positive}"
)
print(
    f"  FN frames: "
    f"{overall_false_negative}"
)
print(
    f"  TN frames: "
    f"{overall_true_negative}"
)
print(
    f"  Precision: "
    f"{overall_vad_metrics['precision']:.4f}"
)
print(
    f"  Recall: "
    f"{overall_vad_metrics['recall']:.4f}"
)
print(
    f"  F1: "
    f"{overall_vad_metrics['f1']:.4f}"
)

ES2002d:
  TP frames: 22554
  FP frames: 416
  FN frames: 3959
  TN frames: 3071
  Precision: 0.9819
  Recall: 0.8507
  F1: 0.9116

ES2008d:
  TP frames: 21694
  FP frames: 1246
  FN frames: 2194
  TN frames: 4866
  Precision: 0.9457
  Recall: 0.9082
  F1: 0.9265

ES2014d:
  TP frames: 10558
  FP frames: 587
  FN frames: 3396
  TN frames: 15459
  Precision: 0.9473
  Recall: 0.7566
  F1: 0.8413

Overall VAD metrics:
  Total frames: 90000
  TP frames: 54806
  FP frames: 2249
  FN frames: 9549
  TN frames: 23396
  Precision: 0.9606
  Recall: 0.8516
  F1: 0.9028


## 15. Compute top-1 speaker-identification accuracy

Top-1 speaker-identification accuracy is calculated only on VAD-detected intervals assigned to the `single_speaker` reference group. The only RTTM speaker associated with each eligible interval is used as its reference identity.

The complete waveform defined by the VAD boundaries has already been passed to ECAPA-TDNN. A prediction is correct when the highest-scoring centroid among all 12 enrolled speakers matches the reference speaker.

In [52]:
speaker_accuracy_by_meeting = {}

for meeting_id in MEETING_IDS:
    meeting_results = serial_results_by_meeting[
        meeting_id
    ]

    eligible_results = [
        result
        for result in meeting_results
        if result["reference_group"]
        == "single_speaker"
    ]

    for result in meeting_results:
        if result["reference_group"] == "single_speaker":
            reference_speaker = (
                result["reference_speakers"][0]
            )

            result["reference_speaker"] = (
                reference_speaker
            )

            result["top1_correct"] = (
                result["top1_speaker"]
                == reference_speaker
            )

        else:
            result["reference_speaker"] = None
            result["top1_correct"] = None

    correct_count = sum(
        result["top1_correct"]
        for result in eligible_results
    )

    eligible_count = len(eligible_results)

    accuracy = (
        correct_count / eligible_count
        if eligible_count > 0
        else 0.0
    )

    speaker_accuracy_by_meeting[meeting_id] = {
        "eligible_count": eligible_count,
        "correct_count": correct_count,
        "incorrect_count": (
            eligible_count - correct_count
        ),
        "accuracy": accuracy,
    }

    print(f"{meeting_id}:")
    print(
        f"  Eligible single-speaker intervals: "
        f"{eligible_count}"
    )
    print(
        f"  Correct top-1 predictions: "
        f"{correct_count}"
    )
    print(
        f"  Incorrect top-1 predictions: "
        f"{eligible_count - correct_count}"
    )
    print(
        f"  Top-1 accuracy: "
        f"{accuracy:.4f}"
    )
    print()


eligible_single_speaker_results = [
    result
    for result in all_serial_results
    if result["reference_group"]
    == "single_speaker"
]

overall_correct_speaker_predictions = sum(
    result["top1_correct"]
    for result in eligible_single_speaker_results
)

overall_eligible_speaker_intervals = len(
    eligible_single_speaker_results
)

overall_speaker_accuracy = (
    overall_correct_speaker_predictions
    / overall_eligible_speaker_intervals
    if overall_eligible_speaker_intervals > 0
    else 0.0
)

print("Overall speaker-identification metrics:")
print(
    f"  Eligible single-speaker intervals: "
    f"{overall_eligible_speaker_intervals}"
)
print(
    f"  Correct top-1 predictions: "
    f"{overall_correct_speaker_predictions}"
)
print(
    f"  Incorrect top-1 predictions: "
    f"{overall_eligible_speaker_intervals - overall_correct_speaker_predictions}"
)
print(
    f"  Top-1 speaker accuracy: "
    f"{overall_speaker_accuracy:.4f}"
)

ES2002d:
  Eligible single-speaker intervals: 56
  Correct top-1 predictions: 54
  Incorrect top-1 predictions: 2
  Top-1 accuracy: 0.9643

ES2008d:
  Eligible single-speaker intervals: 56
  Correct top-1 predictions: 47
  Incorrect top-1 predictions: 9
  Top-1 accuracy: 0.8393

ES2014d:
  Eligible single-speaker intervals: 48
  Correct top-1 predictions: 45
  Incorrect top-1 predictions: 3
  Top-1 accuracy: 0.9375

Overall speaker-identification metrics:
  Eligible single-speaker intervals: 160
  Correct top-1 predictions: 146
  Incorrect top-1 predictions: 14
  Top-1 speaker accuracy: 0.9125


## 16. Compare short and long detected intervals

Eligible single-speaker intervals are divided into short and long groups using the median duration of all 160 eligible VAD-detected intervals. Intervals with duration less than or equal to the median are assigned to the short group, while intervals longer than the median are assigned to the long group.

Top-1 speaker-identification accuracy is calculated separately for the two groups to examine whether short detected utterances are more difficult for ECAPA-TDNN.

In [53]:
eligible_durations = sorted(
    result["duration"]
    for result in eligible_single_speaker_results
)

number_of_durations = len(eligible_durations)
middle_index = number_of_durations // 2

if number_of_durations % 2 == 0:
    median_duration = (
        eligible_durations[middle_index - 1]
        + eligible_durations[middle_index]
    ) / 2
else:
    median_duration = eligible_durations[
        middle_index
    ]


for result in eligible_single_speaker_results:
    if result["duration"] <= median_duration:
        result["duration_group"] = "short"
    else:
        result["duration_group"] = "long"


duration_analysis = {}

print(
    f"Median eligible interval duration: "
    f"{median_duration:.3f} seconds"
)
print()

for duration_group in ["short", "long"]:
    group_results = [
        result
        for result in eligible_single_speaker_results
        if result["duration_group"]
        == duration_group
    ]

    group_count = len(group_results)

    correct_count = sum(
        result["top1_correct"]
        for result in group_results
    )

    incorrect_count = (
        group_count - correct_count
    )

    group_accuracy = (
        correct_count / group_count
        if group_count > 0
        else 0.0
    )

    mean_duration = (
        sum(
            result["duration"]
            for result in group_results
        )
        / group_count
        if group_count > 0
        else 0.0
    )

    duration_analysis[duration_group] = {
        "count": group_count,
        "correct_count": correct_count,
        "incorrect_count": incorrect_count,
        "accuracy": group_accuracy,
        "mean_duration": mean_duration,
    }

    print(f"{duration_group.capitalize()} intervals:")
    print(f"  Count: {group_count}")
    print(
        f"  Mean duration: "
        f"{mean_duration:.3f} seconds"
    )
    print(
        f"  Correct predictions: "
        f"{correct_count}"
    )
    print(
        f"  Incorrect predictions: "
        f"{incorrect_count}"
    )
    print(
        f"  Top-1 accuracy: "
        f"{group_accuracy:.4f}"
    )
    print()

Median eligible interval duration: 1.196 seconds

Short intervals:
  Count: 80
  Mean duration: 0.720 seconds
  Correct predictions: 68
  Incorrect predictions: 12
  Top-1 accuracy: 0.8500

Long intervals:
  Count: 80
  Mean duration: 2.366 seconds
  Correct predictions: 78
  Incorrect predictions: 2
  Top-1 accuracy: 0.9750



## 17. Exploratory analysis of the overlap heuristic

The similarity-margin heuristic is examined only on VAD intervals that contain simultaneous speech from exactly two reference speakers. Consecutive non-overlapping speaker turns and intervals containing more than two reference speakers are excluded.

A successful result requires the heuristic to return two candidate speakers and for both candidates to match the two RTTM reference speakers. This analysis is exploratory and is not treated as a main evaluation metric.

All RTTM-defined single-speaker intervals remain included in the
top-1 speaker-accuracy calculation, regardless of whether the heuristic
returns one or two candidate speakers. Separately, the number of these
intervals for which the heuristic unnecessarily returns a second speaker
is reported as part of the exploratory analysis.

In [54]:
exact_two_speaker_overlap_results = [
    result
    for result in all_serial_results
    if (
        result["reference_group"] == "simultaneous_overlap"
        and len(result["reference_speakers"]) == 2
    )
]

for result in exact_two_speaker_overlap_results:
    result["heuristic_returned_two"] = (
        len(result["candidate_speakers"]) == 2
    )

    result["exact_reference_pair_match"] = (
        set(result["candidate_speakers"])
        == set(result["reference_speakers"])
    )

overlap_returned_two = sum(
    result["heuristic_returned_two"]
    for result in exact_two_speaker_overlap_results
)

overlap_returned_one = (
    len(exact_two_speaker_overlap_results) - overlap_returned_two
)

exact_pair_matches = sum(
    result["exact_reference_pair_match"]
    for result in exact_two_speaker_overlap_results
)

if exact_two_speaker_overlap_results:
    exact_pair_success_rate = (
        exact_pair_matches
        / len(exact_two_speaker_overlap_results)
    )
else:
    exact_pair_success_rate = 0.0

single_speaker_results = [
    result
    for result in all_serial_results
    if result["reference_group"] == "single_speaker"
]

single_speaker_returned_two = sum(
    len(result["candidate_speakers"]) == 2
    for result in single_speaker_results
)

single_speaker_returned_one = (
    len(single_speaker_results) - single_speaker_returned_two
)


overlap_heuristic_summary = {
    "exact_two_speaker_overlap_intervals":
        len(exact_two_speaker_overlap_results),
    "overlap_intervals_returned_two":
        overlap_returned_two,
    "overlap_intervals_returned_one":
        overlap_returned_one,
    "exact_reference_pair_matches":
        exact_pair_matches,
    "exact_pair_success_rate":
        exact_pair_success_rate,
    "single_speaker_intervals":
        len(single_speaker_results),
    "single_speaker_returned_two":
        single_speaker_returned_two,
    "single_speaker_returned_one":
        single_speaker_returned_one,
}


print("RTTM-confirmed exactly two-speaker overlap intervals:")
print(f"  Total intervals: {len(exact_two_speaker_overlap_results)}")
print(f"  Heuristic returned two speakers: {overlap_returned_two}")
print(f"  Heuristic returned one speaker: {overlap_returned_one}")
print(f"  Exact reference-pair matches: {exact_pair_matches}")
print(
    "  Exploratory exact-pair success rate: "
    f"{exact_pair_success_rate:.4f}"
)

print("\nEligible single-speaker intervals:")
print(f"  Total intervals: {len(single_speaker_results)}")
print(f"  Heuristic returned one speaker: {single_speaker_returned_one}")
print(f"  Heuristic returned two speakers: {single_speaker_returned_two}")

RTTM-confirmed exactly two-speaker overlap intervals:
  Total intervals: 59
  Heuristic returned two speakers: 6
  Heuristic returned one speaker: 53
  Exact reference-pair matches: 1
  Exploratory exact-pair success rate: 0.0169

Eligible single-speaker intervals:
  Total intervals: 160
  Heuristic returned one speaker: 132
  Heuristic returned two speakers: 28


## 18. Representative overlap examples

Three representative RTTM-confirmed overlap intervals are inspected:

1. an interval for which both reference speakers were correctly returned;
2. an interval for which two candidates were returned but the pair was incorrect;
3. an interval for which the heuristic returned only one speaker.

The audio is played only for manual inspection and does not affect any
evaluation metric.

In [55]:
correct_pair_examples = [
    result
    for result in exact_two_speaker_overlap_results
    if result["exact_reference_pair_match"]
]

incorrect_pair_examples = [
    result
    for result in exact_two_speaker_overlap_results
    if (
        result["heuristic_returned_two"]
        and not result["exact_reference_pair_match"]
    )
]

missed_overlap_examples = [
    result
    for result in exact_two_speaker_overlap_results
    if not result["heuristic_returned_two"]
]


representative_overlap_examples = [
    ("Correct reference-pair match", correct_pair_examples[0]),
    ("Two candidates but incorrect pair", incorrect_pair_examples[0]),
    ("Overlap missed by the heuristic", missed_overlap_examples[0]),
]


for example_name, result in representative_overlap_examples:
    print(f"\n{example_name}:")
    print(f"  Meeting: {result['meeting']}")
    print(
        "  Time inside evaluation excerpt: "
        f"{result['start']:.2f} to {result['end']:.2f} seconds"
    )
    print(f"  Duration: {result['duration']:.2f} seconds")
    print(f"  Reference speakers: {result['reference_speakers']}")
    print(f"  Returned candidates: {result['candidate_speakers']}")
    print(f"  Similarity margin: {result['similarity_margin']:.4f}")

    audio_segment = evaluation_waveforms[result["meeting"]][
        result["start_sample"]:result["end_sample"]
    ]

    display(
        Audio(
            audio_segment.numpy(),
            rate=TARGET_SAMPLE_RATE
        )
    )


Correct reference-pair match:
  Meeting: ES2008d
  Time inside evaluation excerpt: 225.70 to 230.49 seconds
  Duration: 4.80 seconds
  Reference speakers: ['ES2008d/FEE029', 'ES2008d/FEE030']
  Returned candidates: ['ES2008d/FEE030', 'ES2008d/FEE029']
  Similarity margin: 0.0247



Two candidates but incorrect pair:
  Meeting: ES2002d
  Time inside evaluation excerpt: 74.24 to 75.81 seconds
  Duration: 1.56 seconds
  Reference speakers: ['ES2002d/FEE005', 'ES2002d/MEE008']
  Returned candidates: ['ES2002d/MEE008', 'ES2008d/FEE030']
  Similarity margin: 0.0060



Overlap missed by the heuristic:
  Meeting: ES2002d
  Time inside evaluation excerpt: 5.12 to 7.13 seconds
  Duration: 2.01 seconds
  Reference speakers: ['ES2002d/MEE006', 'ES2002d/MEE007']
  Returned candidates: ['ES2002d/MEE006']
  Similarity margin: 0.2103


## 19. Representative VAD errors

False-positive frames are frames labelled as speech by Silero VAD but
labelled as silence by the RTTM reference. False-negative frames contain
reference speech that was missed by Silero VAD.

Contiguous false-positive and false-negative frames are combined into
error intervals. The two longest intervals of each type are selected for
manual listening. A small amount of surrounding audio is included only
to provide context.

In [56]:
def binary_mask_to_intervals(mask, frame_duration=FRAME_DURATION):
    intervals = []
    start_frame = None

    for frame_index, is_active in enumerate(mask):
        if bool(is_active) and start_frame is None:
            start_frame = frame_index

        elif not bool(is_active) and start_frame is not None:
            start_time = start_frame * frame_duration
            end_time = frame_index * frame_duration

            intervals.append(
                {
                    "start": start_time,
                    "end": end_time,
                    "duration": end_time - start_time,
                }
            )

            start_frame = None

    if start_frame is not None:
        start_time = start_frame * frame_duration
        end_time = len(mask) * frame_duration

        intervals.append(
            {
                "start": start_time,
                "end": end_time,
                "duration": end_time - start_time,
            }
        )

    return intervals


all_false_positive_intervals = []
all_false_negative_intervals = []

for meeting_id in MEETING_IDS:
    reference_mask = np.asarray(
        reference_vad_masks[meeting_id],
        dtype=bool
    )

    predicted_mask = np.asarray(
        predicted_vad_masks[meeting_id],
        dtype=bool
    )

    false_positive_mask = predicted_mask & ~reference_mask
    false_negative_mask = reference_mask & ~predicted_mask

    false_positive_intervals = binary_mask_to_intervals(
        false_positive_mask
    )

    false_negative_intervals = binary_mask_to_intervals(
        false_negative_mask
    )

    for interval in false_positive_intervals:
        interval["meeting"] = meeting_id
        interval["error_type"] = "False positive"
        all_false_positive_intervals.append(interval)

    for interval in false_negative_intervals:
        interval["meeting"] = meeting_id
        interval["error_type"] = "False negative"
        all_false_negative_intervals.append(interval)


representative_false_positives = sorted(
    all_false_positive_intervals,
    key=lambda interval: interval["duration"],
    reverse=True,
)[:2]

representative_false_negatives = sorted(
    all_false_negative_intervals,
    key=lambda interval: interval["duration"],
    reverse=True,
)[:2]

representative_vad_errors = (
    representative_false_positives
    + representative_false_negatives
)


CONTEXT_DURATION = 0.5

for interval in representative_vad_errors:
    context_start = max(
        0.0,
        interval["start"] - CONTEXT_DURATION
    )

    context_end = min(
        EVALUATION_DURATION,
        interval["end"] + CONTEXT_DURATION
    )

    start_sample = int(
        context_start * TARGET_SAMPLE_RATE
    )

    end_sample = int(
        context_end * TARGET_SAMPLE_RATE
    )

    audio_with_context = evaluation_waveforms[interval["meeting"]][
        start_sample:end_sample
    ]

    print(f"\n{interval['error_type']}:")
    print(f"  Meeting: {interval['meeting']}")
    print(
        "  Error interval inside evaluation excerpt: "
        f"{interval['start']:.2f} to {interval['end']:.2f} seconds"
    )
    print(f"  Error duration: {interval['duration']:.2f} seconds")
    print(
        "  Played interval with context: "
        f"{context_start:.2f} to {context_end:.2f} seconds"
    )

    display(
        Audio(
            audio_with_context.numpy(),
            rate=TARGET_SAMPLE_RATE
        )
    )


False positive:
  Meeting: ES2008d
  Error interval inside evaluation excerpt: 224.29 to 225.54 seconds
  Error duration: 1.25 seconds
  Played interval with context: 223.79 to 226.04 seconds



False positive:
  Meeting: ES2008d
  Error interval inside evaluation excerpt: 238.80 to 239.78 seconds
  Error duration: 0.98 seconds
  Played interval with context: 238.30 to 240.28 seconds



False negative:
  Meeting: ES2014d
  Error interval inside evaluation excerpt: 164.23 to 167.02 seconds
  Error duration: 2.79 seconds
  Played interval with context: 163.73 to 167.52 seconds



False negative:
  Meeting: ES2014d
  Error interval inside evaluation excerpt: 184.21 to 186.86 seconds
  Error duration: 2.65 seconds
  Played interval with context: 183.71 to 187.36 seconds


## 20. Lightweight demonstration

The following lightweight demo presents three realistic intervals produced
by the complete serial pipeline:

1. a correctly identified single-speaker interval;
2. an incorrectly identified single-speaker interval;
3. a simultaneous-overlap interval for which the exploratory heuristic
   returned both reference speakers.

The demo plays each detected interval and displays its reference
and predicted speaker information.

In [57]:
demo_correct_single = next(
    result
    for result in eligible_single_speaker_results
    if result["top1_correct"]
)

demo_incorrect_single = next(
    result
    for result in eligible_single_speaker_results
    if not result["top1_correct"]
)

demo_overlap = next(
    result
    for result in exact_two_speaker_overlap_results
    if result["exact_reference_pair_match"]
)


demo_examples = [
    ("Correct single-speaker prediction", demo_correct_single),
    ("Incorrect single-speaker prediction", demo_incorrect_single),
    ("Correct exploratory overlap pair", demo_overlap),
]


for example_name, result in demo_examples:
    print(f"\n{example_name}:")
    print(f"  Meeting: {result['meeting']}")
    print(
        "  Time inside evaluation excerpt: "
        f"{result['start']:.2f} to {result['end']:.2f} seconds"
    )
    print(f"  Duration: {result['duration']:.2f} seconds")
    print(f"  Reference speakers: {result['reference_speakers']}")
    print(f"  Top-1 prediction: {result['top1_speaker']}")
    print(f"  Returned candidates: {result['candidate_speakers']}")
    print(f"  Top-1 cosine score: {result['top1_score']:.4f}")
    print(f"  Similarity margin: {result['similarity_margin']:.4f}")

    audio_segment = evaluation_waveforms[result["meeting"]][
        result["start_sample"]:result["end_sample"]
    ]

    display(
        Audio(
            audio_segment.numpy(),
            rate=TARGET_SAMPLE_RATE
        )
    )


Correct single-speaker prediction:
  Meeting: ES2002d
  Time inside evaluation excerpt: 0.51 to 1.69 seconds
  Duration: 1.18 seconds
  Reference speakers: ['ES2002d/MEE006']
  Top-1 prediction: ES2002d/MEE006
  Returned candidates: ['ES2002d/MEE006']
  Top-1 cosine score: 0.3588
  Similarity margin: 0.2422



Incorrect single-speaker prediction:
  Meeting: ES2002d
  Time inside evaluation excerpt: 44.00 to 44.38 seconds
  Duration: 0.38 seconds
  Reference speakers: ['ES2002d/MEE008']
  Top-1 prediction: ES2008d/MEE031
  Returned candidates: ['ES2008d/MEE031', 'ES2002d/MEE008']
  Top-1 cosine score: 0.1897
  Similarity margin: 0.0217



Correct exploratory overlap pair:
  Meeting: ES2008d
  Time inside evaluation excerpt: 225.70 to 230.49 seconds
  Duration: 4.80 seconds
  Reference speakers: ['ES2008d/FEE029', 'ES2008d/FEE030']
  Top-1 prediction: ES2008d/FEE030
  Returned candidates: ['ES2008d/FEE030', 'ES2008d/FEE029']
  Top-1 cosine score: 0.5038
  Similarity margin: 0.0247
